# TAHAP 1

### Subbab 1.1 — Proses inti

Bagian ini menjalankan: instal pustaka yang dibutuhkan.


In [1]:
# Instal pustaka yang dibutuhkan
!pip install pandas pdfminer.six openpyxl > /dev/null 2>&1
print('✅ Pustaka berhasil diinstal.')

✅ Pustaka berhasil diinstal.


### Subbab 1.2 — Import pustaka

Memuat library yang dibutuhkan untuk tahap ini.


In [2]:
import io
import os
import re
import time
from datetime import date
import pandas as pd
from pdfminer import high_level  # Untuk ekstraksi teks PDF
# from google.colab import drive

### Subbab 1.3 — Proses inti

Bagian ini menjalankan: hubungkan Google Drive.


In [3]:
# Hubungkan Google Drive
# drive.mount('/content/drive')

Mounted at /content/drive


### Subbab 1.4 — Konfigurasi path dan parameter

Mengatur path, parameter, dan variabel penting sebelum proses dijalankan.


In [4]:
# --- Bagian Konfigurasi ---
# !!! PENTING: SESUAIKAN PATH INI DENGAN FOLDER GOOGLE DRIVE ANDA !!!
BASE_DRIVE_PATH = ".." # Changed from colab path to local relative path

# Kata kunci untuk penamaan file output
KEYWORD_FOR_FILENAMING = "Pajak"

# Definisikan path
PATH_RAW_TEXT_OUTPUT    = os.path.join(BASE_DRIVE_PATH, "data/raw")
PATH_PDF_DOWNLOAD       = os.path.join(BASE_DRIVE_PATH, "PDFs_Putusan")  # ← TARUH PDF MANUAL DI SINI
PATH_INITIAL_SCRAPER_CSV = os.path.join(BASE_DRIVE_PATH, "Scraper_CSVs")
PATH_LOGS               = os.path.join(BASE_DRIVE_PATH, "logs")

# Buat direktori jika belum ada
os.makedirs(PATH_RAW_TEXT_OUTPUT, exist_ok=True)
os.makedirs(PATH_PDF_DOWNLOAD, exist_ok=True)
os.makedirs(PATH_INITIAL_SCRAPER_CSV, exist_ok=True)
os.makedirs(PATH_LOGS, exist_ok=True)

CLEANING_LOG_FILE = os.path.join(PATH_LOGS, "cleaning.log")
print(f'✅ Konfigurasi path selesai. Folder PDF: {PATH_PDF_DOWNLOAD}')

✅ Konfigurasi path selesai. Folder PDF: /content/drive/MyDrive/Penalaran Komputer/Penugasan_Sub_CPMK_3/CBR8/PDFs_Putusan


### Subbab 1.5 — Definisi beberapa fungsi

Mendefinisikan beberapa fungsi utama yang dipakai pada tahap ini.


In [5]:
def log_cleaning_action(message, level="INFO"):
    """Menambahkan pesan ke file log pembersihan dengan level."""
    import time
    timestamp = time.strftime('%Y-%m-%d %H:%M:%S')
    log_line = f"{timestamp} [{level}] {message}"
    try:
        with open(CLEANING_LOG_FILE, "a", encoding="utf-8") as f:
            f.write(log_line + "\n")
    except Exception:
        pass  # fail silently if log file not writable
    print(message)


def basic_text_cleaning_ma(text):
    """Pembersihan teks dasar untuk dokumen Mahkamah Agung."""
    if not isinstance(text, str):
        return ""
    text = text.replace("M a h ka m a h A g u n g R e p u blik In d o n esia\n", "")
    text = text.replace("Disclaimer\n", "")
    text = text.replace(
        "Kepaniteraan Mahkamah Agung Republik Indonesia berusaha untuk selalu mencantumkan "
        "informasi paling kini dan akurat sebagai bentuk komitmen Mahkamah Agung untuk "
        "pelayanan publik, transparansi dan akuntabilitas\n", " "
    )
    text = text.replace(
        "pelaksanaan fungsi peradilan. Namun dalam hal-hal tertentu masih dimungkinkan "
        "terjadi permasalahan teknis terkait dengan akurasi dan keterkinian informasi yang "
        "kami sajikan, hal mana akan terus kami perbaiki dari waktu kewaktu.\n", ""
    )
    text = text.replace(
        "Dalam hal Anda menemukan inakurasi informasi yang termuat pada situs ini atau "
        "informasi yang seharusnya ada, namun belum tersedia, maka harap segera hubungi "
        "Kepaniteraan Mahkamah Agung RI melalui :\n", ""
    )
    text = text.replace("Email : kepaniteraan@mahkamahagung.go.id Telp : 021-384 3348 (ext.318)\n", "")
    text = re.sub(r'\s+', ' ', text).strip()
    log_cleaning_action("Performed basic text cleaning (MA disclaimers, whitespace).")
    return text


def convert_pdf_to_cleaned_text(pdf_path_or_stream):
    """Mengekstrak teks dari PDF (path file atau BytesIO) dan menerapkan pembersihan."""
    try:
        raw_text = high_level.extract_text(pdf_path_or_stream)
        return basic_text_cleaning_ma(raw_text)
    except Exception as e:
        log_cleaning_action(f"Error ekstraksi teks PDF: {e}", level="ERROR")
        return ""


print('\u2705 Fungsi pembersihan teks berhasil didefinisikan.')


✅ Fungsi pembersihan teks berhasil didefinisikan.


### Subbab 1.6 — Import pustaka

Memuat library yang dibutuhkan untuk tahap ini.


In [6]:
# List global untuk menyimpan semua data hasil ekstraksi ke satu CSV akhir
all_extracted_data = []
print('✅ Variabel global diinisialisasi.')

✅ Variabel global diinisialisasi.


### Subbab 1.7 — Definisi fungsi `extract_pdf_from_folder`

Mendefinisikan fungsi utama untuk membaca PDF yang sudah diunduh secara manual dari folder `PDFs_Putusan`.


In [7]:
def extract_metadata_from_text(text):
    """
    Mengekstrak metadata dasar dari teks putusan pajak menggunakan regex.
    Mengembalikan dict berisi nomor perkara, tanggal, amar, dll.
    """
    meta = {
        "nomor_perkara": "",
        "tanggal_putusan": "",
        "jenis_perkara": "Pajak",
        "lembaga_peradilan": "",
        "amar_putusan": "",
    }

    # Nomor perkara
    m = re.search(
        r'(?:Nomor|No\.?)\s*[:\.]?\s*([\w\s\/\.\-]+(?:PAJAK|PJK|B|PUT|K|P)[\w\s\/\.\-]+)',
        text, re.IGNORECASE
    )
    if m:
        meta["nomor_perkara"] = m.group(1).strip()[:80]

    # Tanggal putusan
    m = re.search(
        r'(?:dibacakan|diucapkan|ditetapkan|tanggal)[^\n]{0,30}?'
        r'(\d{1,2}\s+(?:Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Desember)\s+\d{4})',
        text, re.IGNORECASE
    )
    if m:
        meta["tanggal_putusan"] = m.group(1).strip()

    # Lembaga peradilan
    m = re.search(
        r'(Mahkamah Agung|Pengadilan Pajak|Pengadilan Tinggi[^,\n]*|Pengadilan Negeri[^,\n]*)',
        text, re.IGNORECASE
    )
    if m:
        meta["lembaga_peradilan"] = m.group(1).strip()

    # Amar putusan (sederhana)
    m = re.search(
        r'MENGADILI,?\s*(.*?)(?:Demikianlah|Memperhatikan pasal)',
        text, re.IGNORECASE | re.DOTALL
    )
    if m:
        meta["amar_putusan"] = re.sub(r'\s+', ' ', m.group(1)).strip()[:300]

    return meta


def extract_pdf_from_folder(pdf_folder_path):
    """
    Membaca semua file PDF dari folder pdf_folder_path,
    mengekstrak teks dan metadata, lalu menyimpan ke file .txt dan list global.
    Menggantikan proses scraping otomatis.
    """
    global all_extracted_data

    pdf_files = [
        f for f in os.listdir(pdf_folder_path)
        if f.lower().endswith('.pdf')
    ]

    if not pdf_files:
        print(f'⚠️  Tidak ada file PDF ditemukan di: {pdf_folder_path}')
        print('    Silakan taruh file PDF putusan ke folder tersebut, lalu jalankan ulang sel ini.')
        return

    print(f'📂 Ditemukan {len(pdf_files)} file PDF. Memulai ekstraksi...\n')

    for idx, pdf_filename in enumerate(sorted(pdf_files), start=1):
        pdf_path = os.path.join(pdf_folder_path, pdf_filename)
        log_cleaning_action(f'[{idx}/{len(pdf_files)}] Memproses: {pdf_filename}')

        # Ekstrak dan bersihkan teks
        full_text = convert_pdf_to_cleaned_text(pdf_path)

        if not full_text or len(full_text) < 100:
            log_cleaning_action(f'  ⚠️  Teks terlalu pendek atau kosong untuk: {pdf_filename}. Dilewati.')
            continue

        # Tambahan cleaning
        full_text = re.sub(r'Halaman\s+\d+\s+dari\s+\d+', '', full_text, flags=re.IGNORECASE)
        full_text = full_text.strip()

        # Ekstrak metadata dari teks
        meta = extract_metadata_from_text(full_text)

        # Buat case_id dan nama file output
        case_id = f'case_{idx:03d}'
        safe_name = re.sub(r'[\\/*?:"<>|]', '_', pdf_filename.replace('.pdf', '').replace('.PDF', ''))
        raw_text_filename = f'{case_id}_{safe_name[:40]}.txt'
        raw_text_filepath = os.path.join(PATH_RAW_TEXT_OUTPUT, raw_text_filename)

        # Simpan teks ke file .txt
        try:
            with open(raw_text_filepath, 'w', encoding='utf-8') as f:
                f.write(full_text)
            log_cleaning_action(f'  ✅ Teks disimpan ke: {raw_text_filename}')
        except Exception as e:
            log_cleaning_action(f'  ❌ Gagal menyimpan teks: {e}')

        # Kumpulkan data ke list
        all_extracted_data.append({
            'case_id'                  : case_id,
            'judul_putusan'            : pdf_filename.replace('.pdf', '').replace('.PDF', ''),
            'nomor_perkara'            : meta['nomor_perkara'],
            'tanggal_putusan'          : meta['tanggal_putusan'],
            'jenis_perkara'            : meta['jenis_perkara'],
            'pasal_digunakan'          : '',
            'nama_pihak'               : '',
            'tingkat_proses'           : '',
            'kata_kunci'               : KEYWORD_FOR_FILENAMING,
            'tahun_dokumen'            : meta['tanggal_putusan'][-4:] if meta['tanggal_putusan'] else '',
            'tanggal_register'         : '',
            'lembaga_peradilan'        : meta['lembaga_peradilan'],
            'amar_putusan'             : meta['amar_putusan'],
            'link_sumber'              : 'MANUAL_DOWNLOAD',
            'link_pdf'                 : pdf_path,
            'nama_file_pdf'            : pdf_filename,
            'nama_file_raw_text'       : raw_text_filename,
            'full_text_putusan_preview': full_text[:200] + '...',
        })

    print(f'\n✅ Selesai! {len(all_extracted_data)} putusan berhasil diekstrak.')


### Subbab 1.8 — Eksekusi: Simpan hasil ke CSV

Menyimpan seluruh data hasil ekstraksi PDF ke file CSV di folder `Scraper_CSVs`.


In [8]:
def save_extracted_data_to_csv(extracted_data_list, keyword, output_folder):
    """
    Menyimpan list data hasil ekstraksi PDF ke file CSV.
    """
    if not extracted_data_list:
        print('⚠️  Tidak ada data untuk disimpan. Pastikan folder PDFs_Putusan sudah berisi PDF.')
        return None

    df = pd.DataFrame(extracted_data_list)
    today_date = date.today().strftime('%Y-%m-%d')
    csv_filename = f'putusan_ma_{keyword.replace(" ", "_")}_{today_date}.csv'
    csv_filepath = os.path.join(output_folder, csv_filename)

    df.to_csv(csv_filepath, index=False, encoding='utf-8')
    print(f'✅ Data disimpan ke: {csv_filepath}')
    print(f'   Total dokumen: {len(df)}')
    return csv_filepath


print('✅ Fungsi penyimpanan CSV berhasil didefinisikan.')

✅ Fungsi penyimpanan CSV berhasil didefinisikan.


### Subbab 1.9 — Eksekusi utama tahap

Menjalankan ekstraksi PDF dari folder `PDFs_Putusan` dan menyimpan hasilnya.


In [9]:
# --- Eksekusi Utama Tahap 1 ---
log_cleaning_action("=" * 60)
log_cleaning_action("TAHAP 1: Membangun Case Base dari PDF Manual — MULAI")
log_cleaning_action("=" * 60)
log_cleaning_action(f"Folder PDF  : {PATH_PDF_DOWNLOAD}")
log_cleaning_action(f"Output teks : {PATH_RAW_TEXT_OUTPUT}")
log_cleaning_action(f"Output CSV  : {PATH_INITIAL_SCRAPER_CSV}")

print('=' * 60)
print('TAHAP 1: Membangun Case Base dari PDF Manual')
print('=' * 60)
print(f'\n\U0001f4c1 Folder PDF  : {PATH_PDF_DOWNLOAD}')
print(f'\U0001f4c1 Output teks : {PATH_RAW_TEXT_OUTPUT}')
print(f'\U0001f4c1 Output CSV  : {PATH_INITIAL_SCRAPER_CSV}')
print()

# Reset list agar tidak duplikat jika sel dijalankan ulang
all_extracted_data = []

# Langkah 1: Ekstrak semua PDF dari folder
extract_pdf_from_folder(PATH_PDF_DOWNLOAD)

# Langkah 2: Simpan hasil ke CSV
if all_extracted_data:
    csv_path = save_extracted_data_to_csv(all_extracted_data, KEYWORD_FOR_FILENAMING, PATH_INITIAL_SCRAPER_CSV)
    log_cleaning_action(f"CSV disimpan: {csv_path}. Total dokumen: {len(all_extracted_data)}")
    print()
    print('--- Preview Data ---')
    df_preview = pd.DataFrame(all_extracted_data)
    display(df_preview[['case_id', 'judul_putusan', 'nomor_perkara', 'tanggal_putusan', 'amar_putusan']].head(10))
else:
    log_cleaning_action("Tidak ada data yang berhasil diekstrak dari PDF.", level="WARNING")
    print('\n\u274c Tidak ada data yang berhasil diekstrak.')
    print('   \u2192 Pastikan file PDF sudah ada di folder PDFs_Putusan.')

log_cleaning_action("TAHAP 1: Membangun Case Base dari PDF Manual — SELESAI")
print('\nTahap 1 selesai.')


TAHAP 1: Membangun Case Base dari PDF Manual — MULAI
Folder PDF  : /content/drive/MyDrive/Penalaran Komputer/Penugasan_Sub_CPMK_3/CBR8/PDFs_Putusan
Output teks : /content/drive/MyDrive/Penalaran Komputer/Penugasan_Sub_CPMK_3/CBR8/data/raw
Output CSV  : /content/drive/MyDrive/Penalaran Komputer/Penugasan_Sub_CPMK_3/CBR8/Scraper_CSVs
TAHAP 1: Membangun Case Base dari PDF Manual

📁 Folder PDF  : /content/drive/MyDrive/Penalaran Komputer/Penugasan_Sub_CPMK_3/CBR8/PDFs_Putusan
📁 Output teks : /content/drive/MyDrive/Penalaran Komputer/Penugasan_Sub_CPMK_3/CBR8/data/raw
📁 Output CSV  : /content/drive/MyDrive/Penalaran Komputer/Penugasan_Sub_CPMK_3/CBR8/Scraper_CSVs

📂 Ditemukan 43 file PDF. Memulai ekstraksi...

[1/43] Memproses: putusan_1111_b_pk_pjk_2017_20250609100401.pdf
Performed basic text cleaning (MA disclaimers, whitespace).
  ✅ Teks disimpan ke: case_001_putusan_1111_b_pk_pjk_2017_2025060910040.txt
[2/43] Memproses: putusan_1227_b_pk_pjk_2017_20250609095335.pdf
Performed basic text 

,case_id,judul_putusan,nomor_perkara,tanggal_putusan,amar_putusan
0,case_001,putusan_1111_b_pk_pjk_2017_20250609100401,1111/B/PK/PJK/2017 PUTUSAN Nomor 1111/B/PK/PJK...,15 Agustus 2014,kembali perkara ini dengan amar sebagaimana ya...
1,case_002,putusan_1227_b_pk_pjk_2017_20250609095335,1227/B/PK/PJK/2017 PUTUSAN Nomor 1227/B/PK/PJK...,5 Agustus 2016,Menolak permohonan peninjauan kembali dari Pem...
2,case_003,putusan_1236_b_pk_pjk_2017_20250609100505,1236/B/PK/PJK/2017PUTUSANNomor 1236/B/PK/PJK/2...,5 Agustus 2016,Menolak permohonan peninjauan kembali dari Pem...
3,case_004,putusan_1245_b_pk_pjk_2017_20250609100435,1245 B/PK/PJK/2017 PUTUSAN Nomor 1245/B/PK/PJK...,10 Agustus 2015,Menolak permohonan peninjauan kembali dari Pem...
4,case_005,putusan_1377_b_pk_pjk_2019_20250609100301,1377/B/PK/Pjk/2019PUTUSANNomor 1377/B/PK/Pjk/2...,13 Agustus 2018,:1.Menolak permohonan peninjauan kembali dari ...
5,case_006,putusan_1477_b_pk_pjk_2017_20250609100220,1477/B/PK/PJK/2017 PUTUSAN Nomor 1477/B/PK/PJK...,15 Januari 2014,Menolak permohonan peninjauan kembali dari Pem...
6,case_007,putusan_1494_b_pk_pjk_2020_20250609095851,1494/B/PK/Pjk/2020 PUTUSAN Nomor 1494/B/PK/Pjk...,30 September 2019,Menolak gugatan Penggugat terhadap Keputusan T...
7,case_008,putusan_1528_b_pk_pjk_2020_20250609100513,1528/B/PK/Pjk/2020 PUTUSAN Nomor 1528/B/PK/Pjk...,23 Agustus 2019,sendiri: 3.1. Menolak permohonan gugatan Termo...
8,case_009,putusan_1539_b_pk_pjk_2018_20250609095746,1539/B/PK/Pjk/2018PUTUSANNomor 1539/B/PK/Pjk/2...,18 Januari 2018,sendiri:3. 1.Menolak permohonan banding Termoh...
9,case_010,putusan_1587_b_pk_pjk_2019_20250609095323,1587/B/PK/Pjk/2019PUTUSANNomor 1587/B/PK/Pjk/2...,29 Januari 2018,SENDIRI:1.Membatalkan Surat Keputusan Direktur...


TAHAP 1: Membangun Case Base dari PDF Manual — SELESAI

Tahap 1 selesai.
